# Soft Diffusion Actor-Critic (SDAC) on MetaWorld MT1 (push-v3)
#
# **Paper:** "Efficient Online Reinforcement Learning for Diffusion Policy"  
# https://arxiv.org/abs/2502.00361 (Ma et al., 2025)
#
# SDAC uses a diffusion model as the policy, trained via **Reweighted Score Matching (RSM)**  
# for max-entropy RL — no backprop through the reverse diffusion chain.
#
# **Key idea:** Weight the denoising score matching loss by `exp(Q(s, a₀) / α)` using  
# Monte Carlo reconstructions from reverse sampling, biasing the score network toward high-value actions.
#
# Implementation: **JAX + Flax + Optax** (matching official repo hyperparameters)

In [ ]:
# ── Installs ──
!pip install -q jax jaxlib dm-haiku optax gymnasium metaworld mujoco wandb python-dotenv imageio[ffmpeg] tqdm

In [ ]:
!apt-get install -y libegl1-mesa-dev libgles2-mesa-dev
!apt-get install -y libosmesa6-dev

In [ ]:
import os
os.environ["MUJOCO_GL"] = "egl"  # or "osmesa" if EGL isn't available
os.environ["MUJOCO_GL"] = "osmesa"

In [ ]:
# ── WandB Login ──
from dotenv import load_dotenv
import os

load_dotenv()
wandb_api_key = os.getenv('WANDB_API_KEY')

!wandb login "{wandb_api_key}"

In [ ]:
# ── Imports ──
import math
import pickle
import time
from dataclasses import dataclass
from functools import partial
from typing import Any, Callable, NamedTuple, Optional, Sequence, Tuple

import haiku as hk
import jax
import jax.numpy as jnp
import jax.tree_util as tree_util
import metaworld
import numpy as np
import optax
import wandb
import imageio
import gymnasium as gym
from gymnasium.spaces import Box
from jax import ShapeDtypeStruct
from tqdm.auto import tqdm

print(f"JAX devices: {jax.devices()}")
print(f"JAX backend: {jax.default_backend()}")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Hyperparameters — matching official SDAC repo (Ma et al., 2025)
# ══════════════════════════════════════════════════════════════════════════════

@dataclass
class Config:
    # ── Environment ──
    env_name: str = "pick-place-v3"
    seed: int = 100

    # ── Training ──
    total_steps: int = 1_000_000
    start_step: int = 10_000              # random exploration before training
    batch_size: int = 256
    buffer_size: int = 1_000_000
    eval_every: int = 10_000              # evaluate every N env steps
    eval_episodes: int = 20
    save_every: int = 100_000              # checkpoint every N steps
    update_log_every: int = 1000          # log update metrics every N update steps

    # ── Diffusion ──
    diffusion_steps: int = 20             # T
    beta_schedule_type: str = "linear"
    beta_schedule_scale: float = 0.8      # betas *= scale
    num_particles: int = 32               # particles for action sampling
    reverse_mc_num: int = 64              # MC samples for importance weights

    # ── Networks ──
    hidden_dim: int = 256
    hidden_num: int = 3                   # MLP depth
    time_embed_dim: int = 16              # sinusoidal time embedding dim

    # ── Optimization ──
    lr: float = 3e-4
    lr_schedule_end: float = 3e-5
    alpha_lr: float = 7e-3               # entropy temperature lr
    gamma: float = 0.99
    tau: float = 0.005                    # target network soft update
    reward_scale: float = 0.2
    noise_scale: float = 0.1

    # ── Update schedule ──
    delay_update: int = 2                 # policy update every N critic updates
    delay_alpha_update: int = 250         # alpha update every N steps
    num_samples: int = 200

    # ── WandB ──
    entity: str = "kkumar1995-new-york-university"
    project_name: str = "ml-research-project"

cfg = Config()
print(f"Config: {cfg}")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Environment Setup — MetaWorld MT1 via MetaWorldWrapper
# ══════════════════════════════════════════════════════════════════════════════

class MetaWorldWrapper:
    """Gymnasium-style wrapper around a MetaWorld environment.

    Normalises the action space to [-1, 1].
    Training instances randomly sample a new task goal on every episode reset.
    Evaluation instances cycle through a fixed task list deterministically.
    """

    def __init__(self, env_cls, tasks, seed=0, action_seed=0, fixed_tasks=False):
        self._env = env_cls()
        self._tasks = list(tasks)
        self._rng = np.random.default_rng(seed)
        self._fixed = fixed_tasks
        self._task_idx = 0
        self._pick_task()

        raw_act = self._env.action_space
        if np.any(raw_act.low != -1.0) or np.any(raw_act.high != 1.0):
            self._needs_rescale = True
            self._act_center = (raw_act.low + raw_act.high) * 0.5
            self._act_half_range = (raw_act.high - raw_act.low) * 0.5
        else:
            self._needs_rescale = False
        self._act_dtype = raw_act.dtype

        self.obs_dim = self._env.observation_space.shape[0]
        self.act_dim = self._env.action_space.shape[0]
        self.observation_space = self._env.observation_space
        self.action_space = Box(
            low=-1.0, high=1.0,
            shape=raw_act.shape,
            dtype=np.float32,
            seed=action_seed,
        )

    def _pick_task(self):
        if self._fixed:
            task = self._tasks[self._task_idx % len(self._tasks)]
            self._task_idx += 1
        else:
            idx = int(self._rng.integers(len(self._tasks)))
            task = self._tasks[idx]
        self._env.set_task(task)

    def reset(self, *, seed=None, options=None):
        self._pick_task()
        obs, info = self._env.reset()
        return obs.astype(np.float32, copy=False), info

    def step(self, action):
        action = np.asarray(action, dtype=self._act_dtype)
        if self._needs_rescale:
            action = action * self._act_half_range + self._act_center
        obs, reward, terminated, truncated, info = self._env.step(action)
        return (
            obs.astype(np.float32, copy=False),
            float(reward),
            bool(terminated),
            bool(truncated),
            info,
        )

    def close(self):
        pass


# ── Seeding ──
def seeding(seed):
    seed_seq = np.random.SeedSequence(seed)
    seed_val = seed_seq.entropy
    bit_generator = np.random.PCG64(seed_val)
    return np.random.Generator(bit_generator), seed_val

master_rng, _ = seeding(cfg.seed)
(env_seed, eval_env_seed, buf_seed,
 net_seed, train_seed) = map(int, master_rng.integers(0, 2**32 - 1, 5))

print(f"Setting up MetaWorld MT1: {cfg.env_name}")
mt1_train = metaworld.MT1(cfg.env_name, seed=env_seed)
mt1_eval  = metaworld.MT1(cfg.env_name, seed=eval_env_seed)

env_cls    = mt1_train.train_classes[cfg.env_name]
train_task = mt1_train.train_tasks[0]
eval_task  = mt1_eval.train_tasks[0]

train_env = MetaWorldWrapper(env_cls, [train_task], seed=env_seed, action_seed=env_seed + 1, fixed_tasks=True)
eval_env  = MetaWorldWrapper(env_cls, [eval_task],  seed=eval_env_seed, action_seed=eval_env_seed + 1, fixed_tasks=True)

obs_dim = train_env.obs_dim
act_dim = train_env.act_dim

print(f"Env: {cfg.env_name} | obs_dim={obs_dim}, act_dim={act_dim}")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Diffusion Utilities — Noise schedule, forward/reverse process
# ══════════════════════════════════════════════════════════════════════════════

@dataclass(frozen=True)
class BetaScheduleCoefficients:
    betas: jax.Array
    alphas: jax.Array
    alphas_cumprod: jax.Array
    alphas_cumprod_prev: jax.Array
    sqrt_alphas_cumprod: jax.Array
    sqrt_one_minus_alphas_cumprod: jax.Array
    log_one_minus_alphas_cumprod: jax.Array
    sqrt_recip_alphas_cumprod: jax.Array
    sqrt_recipm1_alphas_cumprod: jax.Array
    posterior_variance: jax.Array
    posterior_log_variance_clipped: jax.Array
    posterior_mean_coef1: jax.Array
    posterior_mean_coef2: jax.Array

    @staticmethod
    def from_beta(betas):
        alphas = 1.0 - betas
        alphas_cumprod = np.cumprod(alphas, axis=0)
        alphas_cumprod_prev = np.append(1.0, alphas_cumprod[:-1])
        sqrt_alphas_cumprod = np.sqrt(alphas_cumprod)
        sqrt_one_minus_alphas_cumprod = np.sqrt(1.0 - alphas_cumprod)
        log_one_minus_alphas_cumprod = np.log(1.0 - alphas_cumprod)
        sqrt_recip_alphas_cumprod = np.sqrt(1.0 / alphas_cumprod)
        sqrt_recipm1_alphas_cumprod = np.sqrt(1.0 / alphas_cumprod - 1)
        posterior_variance = betas * (1.0 - alphas_cumprod_prev) / (1.0 - alphas_cumprod)
        posterior_log_variance_clipped = np.log(np.maximum(posterior_variance, 1e-20))
        posterior_mean_coef1 = betas * np.sqrt(alphas_cumprod_prev) / (1.0 - alphas_cumprod)
        posterior_mean_coef2 = (1.0 - alphas_cumprod_prev) * np.sqrt(alphas) / (1.0 - alphas_cumprod)
        return BetaScheduleCoefficients(
            *jax.device_put((
                betas, alphas, alphas_cumprod, alphas_cumprod_prev,
                sqrt_alphas_cumprod, sqrt_one_minus_alphas_cumprod,
                log_one_minus_alphas_cumprod, sqrt_recip_alphas_cumprod,
                sqrt_recipm1_alphas_cumprod, posterior_variance,
                posterior_log_variance_clipped, posterior_mean_coef1,
                posterior_mean_coef2,
            ))
        )

    @staticmethod
    def linear_beta_schedule(timesteps, beta_start=1e-4, beta_end=0.999):
        return np.linspace(beta_start, beta_end, timesteps, dtype=np.float64)

    @staticmethod
    def cosine_beta_schedule(timesteps):
        s = 0.008
        t = np.arange(0, timesteps + 1) / timesteps
        alphas_cumprod = np.cos((t + s) / (1 + s) * np.pi / 2) ** 2
        alphas_cumprod /= alphas_cumprod[0]
        betas = 1 - alphas_cumprod[1:] / alphas_cumprod[:-1]
        return np.clip(betas, 0, 0.999)


@dataclass(frozen=True)
class GaussianDiffusion:
    num_timesteps: int
    beta_schedule_scale: float = 0.3
    beta_schedule_type: str = "linear"

    def beta_schedule(self):
        with jax.ensure_compile_time_eval():
            if self.beta_schedule_type == "linear":
                betas = self.beta_schedule_scale * BetaScheduleCoefficients.linear_beta_schedule(self.num_timesteps)
            elif self.beta_schedule_type == "cosine":
                betas = self.beta_schedule_scale * BetaScheduleCoefficients.cosine_beta_schedule(self.num_timesteps)
            else:
                raise ValueError(f"Unknown beta_schedule_type: {self.beta_schedule_type}")
            return BetaScheduleCoefficients.from_beta(betas)

    def p_mean_variance(self, t, x, noise_pred):
        B = self.beta_schedule()
        x_recon = x * B.sqrt_recip_alphas_cumprod[t] - noise_pred * B.sqrt_recipm1_alphas_cumprod[t]
        x_recon = jnp.clip(x_recon, -1, 1)
        model_mean = x_recon * B.posterior_mean_coef1[t] + x * B.posterior_mean_coef2[t]
        model_log_variance = B.posterior_log_variance_clipped[t]
        return model_mean, model_log_variance

    def get_recon(self, t, x, noise):
        B = self.beta_schedule()
        return (x * B.sqrt_recip_alphas_cumprod[t][:, jnp.newaxis]
                - noise * B.sqrt_recipm1_alphas_cumprod[t][:, jnp.newaxis])

    def p_sample(self, key, model, shape):
        x_key, noise_key = jax.random.split(key)
        x = 0.5 * jax.random.normal(x_key, shape)
        noise = jax.random.normal(noise_key, (self.num_timesteps, *shape))

        def body_fn(x, inp):
            t, eps = inp
            noise_pred = model(t, x)
            model_mean, model_log_var = self.p_mean_variance(t, x, noise_pred)
            x = model_mean + (t > 0) * jnp.exp(0.5 * model_log_var) * eps
            return x, None

        t_seq = jnp.arange(self.num_timesteps)[::-1]
        x, _ = jax.lax.scan(body_fn, x, (t_seq, noise))
        return x

    def q_sample(self, t, x_start, noise):
        B = self.beta_schedule()
        return B.sqrt_alphas_cumprod[t] * x_start + B.sqrt_one_minus_alphas_cumprod[t] * noise

    def reverse_samping_weighted_p_loss(self, noise, weights, model, t, x_t):
        if weights.ndim == 1:
            weights = weights.reshape(-1, 1)
        noise_pred = model(t, x_t)
        loss = weights * optax.squared_error(noise_pred, noise)
        return loss.mean()


print(f"Diffusion utilities defined (T={cfg.diffusion_steps}, scale={cfg.beta_schedule_scale})")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Network Architectures — dm-haiku Score Network (Policy) and Q-Network
# ══════════════════════════════════════════════════════════════════════════════

Activation = Callable[[jax.Array], jax.Array]
Identity: Activation = lambda x: x

def mish(x):
    """Mish activation: x * tanh(softplus(x))."""
    return x * jnp.tanh(jax.nn.softplus(x))


def is_broadcastable(src, dst):
    try:
        return jnp.broadcast_shapes(src, dst) == dst
    except ValueError:
        return False


def fix_repr(cls):
    """Remove haiku auto-generated __repr__; keep dataclass repr."""
    del cls.__repr__
    postinit = getattr(cls, "__post_init__")
    def __post_init__(self):
        postinit(self)
        if hk.running_init():
            print(self)
    cls.__post_init__ = __post_init__
    return cls


def mlp(hidden_sizes, output_size, activation, output_activation, *, squeeze_output=False):
    layers = []
    for h in hidden_sizes:
        layers += [hk.Linear(h), activation]
    layers += [hk.Linear(output_size), output_activation]
    if squeeze_output:
        layers.append(partial(jnp.squeeze, axis=-1))
    return hk.Sequential(layers)


def scaled_sinusoidal_encoding(t, *, dim, theta=10000, batch_shape=None):
    assert dim % 2 == 0
    if batch_shape is not None:
        assert is_broadcastable(jnp.shape(t), batch_shape)
    scale = 1 / dim ** 0.5
    half_dim = dim // 2
    freq_seq = jnp.arange(half_dim) / half_dim
    inv_freq = theta ** -freq_seq
    emb = jnp.einsum("..., j -> ... j", t, inv_freq)
    emb = jnp.concatenate((jnp.sin(emb), jnp.cos(emb)), axis=-1) * scale
    if batch_shape is not None:
        emb = jnp.broadcast_to(emb, (*batch_shape, dim))
    return emb


@dataclass
@fix_repr
class QNet(hk.Module):
    hidden_sizes: Sequence[int]
    activation: Activation
    output_activation: Activation = Identity
    name: str = None

    def __call__(self, obs, act):
        inp = jnp.concatenate((obs, act), axis=-1)
        return mlp(self.hidden_sizes, 1, self.activation, self.output_activation,
                   squeeze_output=True)(inp)


@dataclass
@fix_repr
class DACERPolicyNet(hk.Module):
    hidden_sizes: Sequence[int]
    activation: Activation
    output_activation: Activation = Identity
    time_dim: int = 16
    name: str = None

    def __call__(self, obs, act, t):
        act_dim = act.shape[-1]
        te = scaled_sinusoidal_encoding(t, dim=self.time_dim, batch_shape=obs.shape[:-1])
        te = hk.Linear(self.time_dim * 2)(te)
        te = self.activation(te)
        te = hk.Linear(self.time_dim)(te)
        inp = jnp.concatenate((obs, act, te), axis=-1)
        return mlp(self.hidden_sizes, act_dim, self.activation, self.output_activation)(inp)


print("Networks: DACERPolicyNet + QNet (dm-haiku, Mish activation)")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Replay Buffer (TreeBuffer) + Experience
# ══════════════════════════════════════════════════════════════════════════════

class Experience(NamedTuple):
    obs: Any
    action: Any
    reward: Any
    done: Any
    next_obs: Any

    def batch_size(self):
        try:
            if self.reward.ndim > 0:
                return self.reward.shape[0]
        except AttributeError:
            pass
        return None

    @staticmethod
    def create_example(obs_dim, action_dim, batch_size=None):
        leading = (batch_size,) if batch_size is not None else ()
        return Experience(
            obs=np.zeros((*leading, obs_dim), dtype=np.float32),
            action=np.zeros((*leading, action_dim), dtype=np.float32),
            reward=np.zeros(leading, dtype=np.float32),
            next_obs=np.zeros((*leading, obs_dim), dtype=np.float32),
            done=np.zeros(leading, dtype=np.bool_),
        )

    @staticmethod
    def create(obs, action, reward, terminated, truncated, next_obs, info=None):
        return Experience(obs=obs, action=action, reward=reward,
                          done=terminated, next_obs=next_obs)


class TreeBuffer:
    def __init__(self, spec, size, seed=0):
        def make(sd):
            return np.empty((size, *sd.shape), dtype=sd.dtype)
        leaves, treedef = tree_util.tree_flatten(spec)
        self.buffers = tuple(make(sd) for sd in leaves)
        self.treedef = treedef
        self.rng = np.random.default_rng(seed)
        self.max_len = size
        self.len = 0
        self.ptr = 0

    def add(self, sample):
        leaves = self.treedef.flatten_up_to(sample)
        for leaf, buf in zip(leaves, self.buffers):
            buf[self.ptr] = leaf
        self._advance()

    def sample(self, size):
        indices = self.rng.integers(0, self.len, size=size)
        leaves = tuple(np.take(buf, indices, axis=0) for buf in self.buffers)
        return tree_util.tree_unflatten(self.treedef, leaves)

    def _advance(self, size=1):
        self.len = min(self.len + size, self.max_len)
        self.ptr = (self.ptr + size) % self.max_len

    def __len__(self):
        return self.len

    @staticmethod
    def from_experience(obs_dim, act_dim, size, seed=0):
        example = Experience.create_example(obs_dim, act_dim)
        def to_spec(x):
            return ShapeDtypeStruct(x.shape, x.dtype)
        spec = tree_util.tree_map(to_spec, example)
        return TreeBuffer(spec, size, seed)


buffer = TreeBuffer.from_experience(obs_dim, act_dim, size=cfg.buffer_size, seed=buf_seed)
print(f"TreeBuffer created: capacity={cfg.buffer_size:,}")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Diffv2Net + SDAC Algorithm
# ══════════════════════════════════════════════════════════════════════════════

def random_key_from_data(data):
    mean = jnp.mean(data)
    std = jnp.std(data)
    seed = (mean * std).view(jnp.uint32)
    return jax.random.key(seed)


class Diffv2Params(NamedTuple):
    q1: hk.Params
    q2: hk.Params
    target_q1: hk.Params
    target_q2: hk.Params
    policy: hk.Params
    target_poicy: hk.Params   # intentional typo kept from original repo
    log_alpha: jax.Array


@dataclass
class Diffv2Net:
    q: Callable
    policy: Callable
    num_timesteps: int
    act_dim: int
    num_particles: int
    target_entropy: float
    noise_scale: float
    beta_schedule_scale: float
    beta_schedule_type: str = "linear"

    @property
    def diffusion(self):
        return GaussianDiffusion(self.num_timesteps,
                                 self.beta_schedule_scale,
                                 self.beta_schedule_type)

    def get_action(self, key, policy_params, obs):
        policy_params, log_alpha, q1_params, q2_params = policy_params

        def model_fn(t, x):
            return self.policy(policy_params, obs, x, t)

        def sample(k):
            act = self.diffusion.p_sample(k, model_fn, (*obs.shape[:-1], self.act_dim))
            q = jnp.minimum(self.q(q1_params, obs, act), self.q(q2_params, obs, act))
            return act.clip(-1, 1), q

        key, noise_key = jax.random.split(key)
        if self.num_particles == 1:
            act, _ = sample(key)
        else:
            keys = jax.random.split(key, self.num_particles)
            acts, qs = jax.vmap(sample)(keys)
            best = jnp.argmax(qs, axis=0, keepdims=True)
            act = jnp.take_along_axis(acts, best[..., None], axis=0).squeeze(axis=0)
        act = act + jax.random.normal(noise_key, act.shape) * jnp.exp(log_alpha) * self.noise_scale
        return act

    def get_deterministic_action(self, policy_params, obs):
        key = random_key_from_data(obs)
        policy_params_tuple, log_alpha, q1_params, q2_params = policy_params
        silent_log_alpha = -jnp.inf
        return self.get_action(key, (policy_params_tuple, silent_log_alpha, q1_params, q2_params), obs)


def create_diffv2_net(
    key, obs_dim, act_dim,
    hidden_sizes, diffusion_hidden_sizes,
    activation=jax.nn.relu,
    num_timesteps=20,
    num_particles=4,
    noise_scale=0.05,
    target_entropy_scale=0.9,
    beta_schedule_scale=0.3,
    beta_schedule_type="linear",
):
    q_net = hk.without_apply_rng(hk.transform(
        lambda obs, act: QNet(hidden_sizes, activation)(obs, act)
    ))
    policy_net = hk.without_apply_rng(hk.transform(
        lambda obs, act, t: DACERPolicyNet(diffusion_hidden_sizes, activation)(obs, act, t)
    ))

    @jax.jit
    def init(key, obs, act):
        k1, k2, k3 = jax.random.split(key, 3)
        q1 = q_net.init(k1, obs, act)
        q2 = q_net.init(k2, obs, act)
        pol = policy_net.init(k3, obs, act, 0)
        log_alpha = jnp.array(math.log(5), dtype=jnp.float32)
        return Diffv2Params(q1, q2, q1, q2, pol, pol, log_alpha)

    sample_obs = jnp.zeros((1, obs_dim))
    sample_act = jnp.zeros((1, act_dim))
    params = init(key, sample_obs, sample_act)

    net = Diffv2Net(
        q=q_net.apply,
        policy=policy_net.apply,
        num_timesteps=num_timesteps,
        act_dim=act_dim,
        target_entropy=-act_dim * target_entropy_scale,
        num_particles=num_particles,
        noise_scale=noise_scale,
        beta_schedule_scale=beta_schedule_scale,
        beta_schedule_type=beta_schedule_type,
    )
    return net, params


# ── Algorithm base class ──
class Algorithm:
    def _implement_common_behavior(self, stateless_update, stateless_get_action,
                                   stateless_get_deterministic_action, stateless_get_value=None):
        self._update = jax.jit(stateless_update)
        self._get_action = jax.jit(stateless_get_action)
        self._get_deterministic_action = jax.jit(stateless_get_deterministic_action)
        if stateless_get_value is not None:
            self._get_value = jax.jit(stateless_get_value)

    def update(self, key, data):
        self.state, info = self._update(key, self.state, data)
        scalar = {k: float(v) for k, v in info.items() if not k.startswith("hist")}
        hist = {k: v for k, v in info.items() if k.startswith("hist")}
        return scalar, hist

    def get_action(self, key, obs):
        action = self._get_action(key, self.get_policy_params(), obs)
        return np.asarray(action)

    def get_deterministic_action(self, obs):
        action = self._get_deterministic_action(self.get_policy_params(), obs)
        return np.asarray(action)

    def save(self, path):
        state = jax.device_get(self.state)
        with open(path, "wb") as f:
            pickle.dump(state, f)

    def load(self, path):
        with open(path, "rb") as f:
            state = pickle.load(f)
        self.state = jax.device_put(state)

    def save_policy(self, path):
        policy = jax.device_get(self.get_policy_params())
        with open(path, "wb") as f:
            pickle.dump(policy, f)

    def save_q(self, path):
        value = jax.device_get(self.get_value_params())
        with open(path, "wb") as f:
            pickle.dump(value, f)

    def get_policy_params(self):
        return self.state.params.policy

    def get_value_params(self):
        return self.state.params.q1, self.state.params.q2

    def warmup(self, data):
        key = jax.random.key(0)
        obs = data.obs[0]
        self._update(key, self.state, data)
        self._get_action(key, self.get_policy_params(), obs)
        self._get_deterministic_action(self.get_policy_params(), obs)


# ── SDAC-specific types ──
class Diffv2OptStates(NamedTuple):
    q1: optax.OptState
    q2: optax.OptState
    policy: optax.OptState
    log_alpha: optax.OptState

class Diffv2TrainState(NamedTuple):
    params: Diffv2Params
    opt_state: Diffv2OptStates
    step: int
    entropy: float
    running_mean: float
    running_std: float


class SDAC(Algorithm):

    def __init__(
        self, agent, params, *,
        gamma=0.99, lr=1e-4, alpha_lr=3e-2,
        lr_schedule_end=5e-5, tau=0.005,
        delay_alpha_update=250, delay_update=2,
        reward_scale=0.2, num_samples=200,
    ):
        self.agent = agent
        self.gamma = gamma
        self.tau = tau
        self.delay_alpha_update = delay_alpha_update
        self.delay_update = delay_update
        self.reward_scale = reward_scale
        self.num_samples = num_samples

        self.optim = optax.adam(lr)
        lr_schedule = optax.schedules.linear_schedule(
            init_value=lr,
            end_value=lr_schedule_end,
            transition_steps=int(5e4),
            transition_begin=int(2.5e4),
        )
        self.policy_optim = optax.adam(learning_rate=lr_schedule)
        self.alpha_optim = optax.adam(alpha_lr)

        self.state = Diffv2TrainState(
            params=params,
            opt_state=Diffv2OptStates(
                q1=self.optim.init(params.q1),
                q2=self.optim.init(params.q2),
                policy=self.policy_optim.init(params.policy),
                log_alpha=self.alpha_optim.init(params.log_alpha),
            ),
            step=jnp.int32(0),
            entropy=jnp.float32(0.0),
            running_mean=jnp.float32(0.0),
            running_std=jnp.float32(1.0),
        )

        # ── Stateless update (will be JIT-compiled) ──
        @jax.jit
        def stateless_update(key, state, data):
            obs, action, reward, next_obs, done = (
                data.obs, data.action, data.reward, data.next_obs, data.done
            )
            q1_params, q2_params, tq1, tq2, policy_params, tpol, log_alpha = state.params
            q1_os, q2_os, pol_os, la_os = state.opt_state
            step = state.step
            running_mean = state.running_mean
            running_std = state.running_std

            (next_eval_key, new_eval_key, _nq1, _nq2,
             _la_key, dt_key, dn_key) = jax.random.split(key, 7)

            reward = reward * self.reward_scale

            def get_min_q(s, a):
                return jnp.minimum(agent.q(q1_params, s, a), agent.q(q2_params, s, a))

            def get_min_target_q(s, a):
                return jnp.minimum(agent.q(tq1, s, a), agent.q(tq2, s, a))

            # --- Q target ---
            next_action = agent.get_action(
                next_eval_key, (policy_params, log_alpha, q1_params, q2_params), next_obs
            )
            q_target = jnp.minimum(agent.q(tq1, next_obs, next_action),
                                   agent.q(tq2, next_obs, next_action))
            q_backup = reward + (1 - done) * self.gamma * q_target

            # --- Q losses ---
            def q_loss_fn(qp):
                q = agent.q(qp, obs, action)
                return jnp.mean((q - q_backup) ** 2), q

            (q1_loss, q1_val), q1_grads = jax.value_and_grad(q_loss_fn, has_aux=True)(q1_params)
            (q2_loss, q2_val), q2_grads = jax.value_and_grad(q_loss_fn, has_aux=True)(q2_params)

            # --- Policy loss (reverse-sampling weighted) ---
            new_action = agent.get_action(
                new_eval_key, (policy_params, log_alpha, q1_params, q2_params), obs
            )
            diff_key1, diff_key2 = jax.random.split(dn_key, 2)
            t = jax.random.randint(dt_key, (obs.shape[0],), 0, agent.num_timesteps)
            noise1 = jax.random.normal(diff_key1, action.shape)
            tilde_at = jax.vmap(agent.diffusion.q_sample)(t, new_action, noise1)

            reverse_mc_num = 64
            tilde_at = jnp.repeat(tilde_at, reverse_mc_num, axis=0)
            t_rep = jnp.repeat(t, reverse_mc_num, axis=0)
            wide_obs = jnp.repeat(obs, reverse_mc_num, axis=0)

            def policy_loss_fn(pp):
                def denoiser(tt, xx):
                    return agent.policy(pp, wide_obs, xx, tt)

                noise2 = jax.random.normal(diff_key2,
                                           (action.shape[0] * reverse_mc_num, action.shape[1]))
                recon = agent.diffusion.get_recon(t_rep, tilde_at, noise2).clip(-1, 1)
                q_min = get_min_q(wide_obs, recon) * 5.0 / jnp.exp(log_alpha)
                q_mean = q_min.mean()
                q_std = q_min.std()
                q_reshape = q_min.reshape((-1, reverse_mc_num))
                Z = jax.nn.logsumexp(q_reshape, axis=1, keepdims=True)
                q_weights = jnp.exp(q_reshape - Z).flatten()

                loss = agent.diffusion.reverse_samping_weighted_p_loss(
                    noise2, q_weights, denoiser, t_rep, tilde_at
                )
                return loss, (q_weights, q_min, q_mean, q_std, recon)

            (total_loss, (q_weights, scaled_q, q_mean, q_std, recon)), pol_grads = (
                jax.value_and_grad(policy_loss_fn, has_aux=True)(policy_params)
            )

            # --- Alpha loss ---
            def log_alpha_loss_fn(la):
                approx_ent = 0.5 * agent.act_dim * jnp.log(
                    2 * jnp.pi * jnp.exp(1) * (0.1 * jnp.exp(la)) ** 2
                )
                return -la * (-jax.lax.stop_gradient(approx_ent) + agent.target_entropy)

            # --- Param updates ---
            def param_update(optim, params, grads, opt_state):
                updates, new_os = optim.update(grads, opt_state)
                return optax.apply_updates(params, updates), new_os

            def delay_param_update(optim, params, grads, opt_state):
                return jax.lax.cond(
                    step % self.delay_update == 0,
                    lambda p, os: param_update(optim, p, grads, os),
                    lambda p, os: (p, os),
                    params, opt_state,
                )

            def delay_alpha_update_fn(optim, params, opt_state):
                return jax.lax.cond(
                    step % self.delay_alpha_update == 0,
                    lambda p, os: param_update(optim, p,
                                               jax.grad(log_alpha_loss_fn)(p), os),
                    lambda p, os: (p, os),
                    params, opt_state,
                )

            def delay_target_update(params, target, tau):
                return jax.lax.cond(
                    step % self.delay_update == 0,
                    lambda tgt: optax.incremental_update(params, tgt, tau),
                    lambda tgt: tgt,
                    target,
                )

            q1_params, q1_os = param_update(self.optim, q1_params, q1_grads, q1_os)
            q2_params, q2_os = param_update(self.optim, q2_params, q2_grads, q2_os)
            policy_params, pol_os = delay_param_update(self.policy_optim, policy_params, pol_grads, pol_os)
            log_alpha, la_os = delay_alpha_update_fn(self.alpha_optim, log_alpha, la_os)

            tq1 = delay_target_update(q1_params, tq1, self.tau)
            tq2 = delay_target_update(q2_params, tq2, self.tau)
            tpol = delay_target_update(policy_params, tpol, self.tau)

            new_rm = running_mean + 0.001 * (q_mean - running_mean)
            new_rs = running_std + 0.001 * (q_std - running_std)

            new_state = Diffv2TrainState(
                params=Diffv2Params(q1_params, q2_params, tq1, tq2, policy_params, tpol, log_alpha),
                opt_state=Diffv2OptStates(q1=q1_os, q2=q2_os, policy=pol_os, log_alpha=la_os),
                step=step + 1,
                entropy=jnp.float32(0.0),
                running_mean=new_rm,
                running_std=new_rs,
            )
            info = {
                "q1_loss": q1_loss,
                "q1_mean": jnp.mean(q1_val),
                "q2_loss": q2_loss,
                "policy_loss": total_loss,
                "alpha": jnp.exp(log_alpha),
                "hist_q_weights": q_weights,
                "hist_t": t_rep,
                "scale_q_mean": jnp.mean(scaled_q),
                "running_q_mean": new_rm,
                "running_q_std": new_rs,
            }
            return new_state, info

        self._implement_common_behavior(
            stateless_update,
            agent.get_action,
            agent.get_deterministic_action,
            stateless_get_value=agent.q,
        )

    # Override base-class methods for correct SDAC policy params
    def get_policy_params(self):
        p = self.state.params
        return (p.policy, p.log_alpha, p.q1, p.q2)

    def get_eval_policy_params(self):
        p = self.state.params
        return (p.target_poicy, p.log_alpha, p.q1, p.q2)

    def get_action(self, key, obs):
        action = self._get_action(key, self.get_policy_params(), obs)
        return np.asarray(action)

    def get_deterministic_action(self, obs):
        action = self._get_deterministic_action(self.get_eval_policy_params(), obs)
        return np.asarray(action)

    def warmup(self, data):
        key = jax.random.key(0)
        obs = data.obs[0]
        self._update(key, self.state, data)
        self._get_action(key, self.get_policy_params(), obs)
        self._get_deterministic_action(self.get_eval_policy_params(), obs)

    def get_value_params(self):
        return self.state.params.q1, self.state.params.q2


# ── Create network + algorithm ──
init_key = jax.random.key(net_seed)
train_key = jax.random.key(train_seed)

hidden_sizes = [cfg.hidden_dim] * cfg.hidden_num
diff_hidden  = [cfg.hidden_dim] * cfg.hidden_num

agent, params = create_diffv2_net(
    init_key, obs_dim, act_dim,
    hidden_sizes, diff_hidden, mish,
    num_timesteps=cfg.diffusion_steps,
    num_particles=cfg.num_particles,
    noise_scale=cfg.noise_scale,
    beta_schedule_scale=cfg.beta_schedule_scale,
    beta_schedule_type=cfg.beta_schedule_type,
)

algorithm = SDAC(
    agent, params,
    lr=cfg.lr,
    alpha_lr=cfg.alpha_lr,
    delay_alpha_update=cfg.delay_alpha_update,
    lr_schedule_end=cfg.lr_schedule_end,
)

print(f"SDAC algorithm created.")
print(f"Policy params: {sum(x.size for x in jax.tree.leaves(params.policy)):,}")
print(f"Q1 params: {sum(x.size for x in jax.tree.leaves(params.q1)):,}")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Evaluation + Random Warmup Helpers
# ══════════════════════════════════════════════════════════════════════════════

def evaluate(eval_env, algorithm, num_episodes):
    """Run num_episodes with the deterministic policy; return summary metrics."""
    returns, lengths, successes = [], [], []
    for _ in range(num_episodes):
        obs, _ = eval_env.reset()
        ep_ret = 0.0
        ep_len = 0
        success = False
        while True:
            action = algorithm.get_deterministic_action(obs)
            obs, reward, terminated, truncated, info = eval_env.step(action)
            ep_ret += reward
            ep_len += 1
            if info.get("success", False):
                success = True
            if terminated or truncated:
                break
        returns.append(ep_ret)
        lengths.append(ep_len)
        successes.append(float(success))
    return {
        "eval/episode_return": float(np.mean(returns)),
        "eval/episode_return_std": float(np.std(returns)),
        "eval/episode_length": float(np.mean(lengths)),
        "eval/success_rate": float(np.mean(successes)),
    }


def random_warmup(env, buffer, start_step):
    """Fill buffer with start_step random transitions."""
    obs, _ = env.reset()
    for _ in range(start_step):
        action = env.action_space.sample()
        next_obs, reward, terminated, truncated, _ = env.step(action)
        buffer.add(Experience.create(obs, action, reward, terminated, truncated, next_obs))
        if terminated or truncated:
            obs, _ = env.reset()
        else:
            obs = next_obs
    return obs


print("Evaluation and warmup functions defined.")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Generate Video/GIF of Trained Agent
# ══════════════════════════════════════════════════════════════════════════════

def record_agent_video(
    algorithm, rng_key,
    num_episodes=3, output_path="sdac_push_v3.gif", fps=30,
):
    """Record the trained agent and save as a GIF."""
    render_env = gym.make(
        "Meta-World/MT1",
        env_name=cfg.env_name,
        seed=cfg.seed,
        render_mode="rgb_array",
        camera_name="corner2",  # try: "corner", "corner2", "corner3", "behindGripper", "gripperPOV"
    )

    all_frames = []

    for ep in range(num_episodes):
        obs, info = render_env.reset()
        done = False
        truncated = False
        ep_reward = 0.0
        success = False

        while not (done or truncated):
            frame = render_env.render()
            all_frames.append(frame[::-1])  # flip vertically

            action = algorithm.get_deterministic_action(obs)
            obs, reward, done, truncated, info = render_env.step(action)
            ep_reward += reward
            if info.get("success", False):
                success = True

        print(f"  Episode {ep+1}: reward={ep_reward:.1f}, success={success}")

    render_env.close()

    imageio.mimsave(output_path, all_frames, fps=fps, loop=0)
    print(f"\nSaved {len(all_frames)} frames to {output_path}")

    wandb.log({"eval/video": wandb.Video(output_path, fps=fps, format="gif")})
    print("Video logged to WandB.")
    return output_path

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Training Loop
# ══════════════════════════════════════════════════════════════════════════════

import os

# Initialize WandB
wandb.init(
    entity=cfg.entity,
    project=cfg.project_name,
    name=f"sdac-{cfg.env_name}-T{cfg.diffusion_steps}-alpha{cfg.alpha_lr}-s{cfg.seed}",
    config={
        "env_name": cfg.env_name,
        "total_steps": cfg.total_steps,
        "start_step": cfg.start_step,
        "batch_size": cfg.batch_size,
        "buffer_size": cfg.buffer_size,
        "diffusion_steps": cfg.diffusion_steps,
        "beta_schedule_type": cfg.beta_schedule_type,
        "beta_schedule_scale": cfg.beta_schedule_scale,
        "num_particles": cfg.num_particles,
        "reverse_mc_num": cfg.reverse_mc_num,
        "hidden_dim": cfg.hidden_dim,
        "hidden_num": cfg.hidden_num,
        "lr": cfg.lr,
        "lr_schedule_end": cfg.lr_schedule_end,
        "alpha_lr": cfg.alpha_lr,
        "gamma": cfg.gamma,
        "tau": cfg.tau,
        "reward_scale": cfg.reward_scale,
        "delay_update": cfg.delay_update,
        "delay_alpha_update": cfg.delay_alpha_update,
        "noise_scale": cfg.noise_scale,
        "seed": cfg.seed,
        "obs_dim": obs_dim,
        "act_dim": act_dim,
    },
)

# ── JIT warmup ──
print("JIT tracing ...")
dummy = Experience.create_example(obs_dim, act_dim, cfg.batch_size)
algorithm.warmup(dummy)

# ── Random warmup phase ──
print(f"Random warmup: {cfg.start_step:,} steps ...")
obs = random_warmup(train_env, buffer, cfg.start_step)

# ── Training state ──
print(f"Training: {cfg.total_steps:,} steps ...")
sample_step = 0
update_step = 0

ep_ret_buf, ep_len_buf, ep_suc_buf = [], [], []
cur_ret, cur_len, cur_suc = 0.0, 0, False
update_acc = {k: [] for k in ("q1_loss", "q2_loss", "policy_loss", "alpha")}

progress = tqdm(total=cfg.total_steps, desc="Steps", dynamic_ncols=True)
iter_key = jax.jit(lambda s: jax.random.fold_in(train_key, s))

start_time = time.time()

while sample_step < cfg.total_steps:

    # ── Sample ──
    step_key = iter_key(sample_step)
    action = algorithm.get_action(step_key, obs)
    next_obs, reward, terminated, truncated, info = train_env.step(action)

    buffer.add(Experience.create(obs, action, reward, terminated, truncated, next_obs))

    cur_ret += reward
    cur_len += 1
    if info.get("success", False):
        cur_suc = True

    if terminated or truncated:
        ep_ret_buf.append(cur_ret)
        ep_len_buf.append(cur_len)
        ep_suc_buf.append(float(cur_suc))
        cur_ret, cur_len, cur_suc = 0.0, 0, False
        obs, _ = train_env.reset()

        if len(ep_ret_buf) >= 10:
            wandb.log({
                "train/avg_return": float(np.mean(ep_ret_buf)),
                "train/avg_length": float(np.mean(ep_len_buf)),
                "train/success_rate": float(np.mean(ep_suc_buf)),
            }, step=sample_step)
            ep_ret_buf.clear(); ep_len_buf.clear(); ep_suc_buf.clear()
    else:
        obs = next_obs

    sample_step += 1
    progress.update(1)

    # ── Update ──
    upd_key = jax.random.fold_in(train_key, sample_step + int(1e9))
    data = buffer.sample(cfg.batch_size)
    scalar_info, _ = algorithm.update(upd_key, data)
    update_step += 1

    for k in update_acc:
        update_acc[k].append(scalar_info[k])

    if update_step % cfg.update_log_every == 0:
        wandb.log({
            "train/q1_loss": float(np.mean(update_acc["q1_loss"])),
            "train/q2_loss": float(np.mean(update_acc["q2_loss"])),
            "train/policy_loss": float(np.mean(update_acc["policy_loss"])),
            "train/alpha": float(np.mean(update_acc["alpha"])),
            "train/update_step": update_step,
            "train/fps": sample_step / (time.time() - start_time),
        }, step=sample_step)
        for k in update_acc:
            update_acc[k].clear()

    # ── Evaluation ──
    if sample_step % cfg.eval_every == 0:
        metrics = evaluate(eval_env, algorithm, cfg.eval_episodes)
        wandb.log(metrics, step=sample_step)
        progress.set_postfix({
            "ret": f"{metrics['eval/episode_return']:.1f}",
            "suc": f"{metrics['eval/success_rate']:.2f}",
        })
        print(f"Step {sample_step:>7d} | "
              f"Eval reward: {metrics['eval/episode_return']:.1f} +/- {metrics['eval/episode_return_std']:.1f} | "
              f"Success: {metrics['eval/success_rate']:.2f} | "
              f"Alpha: {float(jnp.exp(algorithm.state.params.log_alpha)):.4f}")

    # ── Checkpoint (save as wandb artifacts) ──
    if sample_step % cfg.save_every == 0:
        policy_path = f"policy-step{sample_step}.pkl"
        value_path  = f"value-step{sample_step}.pkl"
        algorithm.save_policy(policy_path)
        algorithm.save_q(value_path)

        artifact = wandb.Artifact(
            f"sdac-{cfg.env_name}-T{cfg.diffusion_steps}-alpha{cfg.alpha_lr}-s{cfg.seed}-checkpoint",
            type="model",
            description=f"SDAC checkpoint at step {sample_step}",
            metadata={"step": sample_step, "seed": cfg.seed},
        )
        artifact.add_file(policy_path)
        artifact.add_file(value_path)
        wandb.log_artifact(artifact)
        os.remove(policy_path)
        os.remove(value_path)
        print(f"  -> Checkpoint saved to WandB at step {sample_step}")

        # Record the trained agent
        gif_path = record_agent_video(
            algorithm,
            rng_key=jax.random.key(42),
            num_episodes=3, output_path=f"sdac_{cfg.env_name.replace("-","_")}_{sample_step}.gif",
        )
        print(f"\nDone! GIF saved at: {gif_path}")

# ── Final save ──
algorithm.save_policy("policy-final.pkl")
algorithm.save_q("value-final.pkl")

final_artifact = wandb.Artifact(
    f"sdac-{cfg.env_name}-T{cfg.diffusion_steps}-alpha{cfg.alpha_lr}-s{cfg.seed}-final",
    type="model",
    description="SDAC final policy and value networks",
    metadata={"total_steps": cfg.total_steps, "seed": cfg.seed},
)
final_artifact.add_file("policy-final.pkl")
final_artifact.add_file("value-final.pkl")
wandb.log_artifact(final_artifact)
os.remove("policy-final.pkl")
os.remove("value-final.pkl")

progress.close()
elapsed = time.time() - start_time
print(f"\nTraining complete! {cfg.total_steps:,} steps in {elapsed/3600:.1f}h ({cfg.total_steps/elapsed:.0f} fps)")

In [ ]:
# ── Finish WandB run ──
wandb.finish()
print("WandB run finished.")